# PDF to MCQ Walkthrough

A quick guide from PDF upload to MCQs.

## RAG in a Nutshell

We don’t send the whole document to the model. We grab the most relevant parts and use only those.

Flow:
1. Extract text
2. Split into chunks
3. Embed chunks
4. Retrieve top matches
5. Generate questions from that context

## Step 1: Install Dependencies

Install the project packages.

In [1]:
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2: Imports

In [2]:
import os
import io
import json
import re
import shutil
from pathlib import Path
from PyPDF2 import PdfReader

# RAG-specific imports
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

try:
    from pdf2image import convert_from_bytes
    import pytesseract
    OCR_AVAILABLE = True
except ImportError:
    OCR_AVAILABLE = False

print('OCR libraries available:', OCR_AVAILABLE)
print('tesseract binary:', shutil.which('tesseract'))
print('pdftoppm binary:', shutil.which('pdftoppm'))

d:\VS_CODES\Projects\pdf-to-mcq-generator\backend\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OCR libraries available: True
tesseract binary: None
pdftoppm binary: None


## Step 3: Load Env Vars

Loads keys from `.env`.

In [3]:
from dotenv import load_dotenv

env_path = Path('..') / '.env'
load_dotenv(env_path)

GROQ_API_KEY = os.getenv('GROQ_API_KEY')
print('GROQ_API_KEY loaded:', bool(GROQ_API_KEY))

GROQ_API_KEY loaded: True


## Step 4: PDF Path

In [4]:
# Change this to your PDF file path
PDF_PATH = Path('../..') / 'sample.pdf'

print('File exists:', PDF_PATH.exists())
print('File size:', PDF_PATH.stat().st_size, 'bytes')

File exists: True
File size: 61512 bytes


## Step 5: Extract Text

Read the PDF and pull text from each page.

In [5]:
pdf_bytes = PDF_PATH.read_bytes()
reader = PdfReader(io.BytesIO(pdf_bytes))

print('Total pages:', len(reader.pages))

text = ''
for i, page in enumerate(reader.pages):
    page_text = page.extract_text() or ''
    text += page_text + '\n'
    print(f'  Page {i+1}: {len(page_text)} chars')

text = text.strip()
print('\nTotal extracted chars:', len(text))

Total pages: 2
  Page 1: 945 chars
  Page 2: 1023 chars

Total extracted chars: 1961


## Step 6: OCR Fallback

If no text is found, run OCR on the pages.

In [6]:
if not text:
    print('No text from PyPDF2 — trying OCR...')

    if not OCR_AVAILABLE:
        raise RuntimeError('pdf2image / pytesseract not installed')
    if not shutil.which('tesseract') or not shutil.which('pdftoppm'):
        raise RuntimeError('System tools missing: install tesseract-ocr and poppler-utils')

    images = convert_from_bytes(pdf_bytes)
    ocr_parts = []
    for i, img in enumerate(images, 1):
        page_text = pytesseract.image_to_string(img).strip()
        print(f'  OCR page {i}: {len(page_text)} chars')
        if page_text:
            ocr_parts.append(page_text)

    text = '\n'.join(ocr_parts).strip()
    print('\nTotal OCR chars:', len(text))
else:
    print('Text extracted via PyPDF2 — OCR not needed.')

Text extracted via PyPDF2 — OCR not needed.


## Step 7: Preview Text

Sanity‑check the first 1000 characters.

In [7]:
print(text[:1000])

1) Username Validation   
a. only alphabets and digits ,  
b. does not start with a digit   
c. length ≥ 6  
Ex. User123  → Valid, 123User  → Invalid, Us@12  → Invalid  
2) WAP to Sort a dictionary in ascending order of its values.  
Ex.   
Input : {'Math': 70, 'Physics': 85, 'Chemistry': 60}   
output: {'Chemistry': 60, 'Math': 70, 'Physics': 85}  
 
3) WAP that gets password from user. Password must have at least 8 characters 
and one digit.  Raise WeakPasswordError  if invalid, print the messsage " Correct" 
otherwise.  
 
4) WAP to plot a line chart showing temperature variation during a week. Mark the 
lowest and highest temperatue . 
Ex. 
Days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] 
Temperature  = [20, 32, 31, 33, 34, 43, 36] 
 
5) Define a class Square having a private attribute " side". Implement get_side  and 
set_side methods to accees the private attribute from outside of the class.  
 
 
 
 
 
 
 
 
 
 
 
 
1) 1.Reverse Each Word  
Ex. “Python is Easy ” → “noht

## Step 8: Retrieve Context

Build embeddings, index chunks, and pull the best matches.

In [ ]:
# 1. Initialize the Embeddings Model
# We use a local model from Hugging Face to turn text into numerical vectors.
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("Embedding model loaded.")

# 2. Split the Text into Chunks
# We break the document into smaller pieces to make it easier to search.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)
docs = splitter.create_documents([text])
print(f"Text split into {len(docs)} chunks.")

# 3. Create a FAISS Vector Store
# This creates a searchable in-memory index of our text chunks.
vector_store = FAISS.from_documents(docs, embeddings)
print("FAISS vector store created.")

# 4. Find Relevant Context
# We search the index for chunks that are most similar to our query.
query = "Key concepts, definitions, explanations, and important topics from the document."
retrieved_docs = vector_store.similarity_search(query, k=4) # Get top 4 chunks

retrieved_context = "\n\n".join(doc.page_content for doc in retrieved_docs).strip()

print(f"\nRetrieved context length: {len(retrieved_context)} chars")
print("\n--- Start of Retrieved Context ---")
print(retrieved_context)
print("--- End of Retrieved Context ---\n")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4701.16it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded.
Text split into 3 chunks.
FAISS vector store created.

Retrieved context length: 2204 chars

--- Start of Retrieved Context ---
set_side methods to accees the private attribute from outside of the class.  
 
 
 
 
 
 
 
 
 
 
 
 
1) 1.Reverse Each Word  
Ex. “Python is Easy ” → “nohtyP si ysaE ” 
 
2) WAP to find tuples which have all elements divisible by K  from a list of tuples.  
Ex.  
tuple_list = [(2, 4, 6), (3, 6, 9), (4, 8, 12), (5, 10, 15)] , K = 2  
Tuples divisible by 2 : [(2, 4, 6), (4, 8, 12)]  
 
3) WAP to write first 100 prime numbers  to a file named primenumbers.txt  
 
4) WAP to create a detailed pie chart  representing the monthly expenditure 
distribution of a family. The chart should display labels , percentage values , 
title, different colors  for each slice, and explode one slice to highlight the 
highest expense.  
Categories  = ["Rent", "Food", "Education" , "Transport" , "Entertainment" , "Savings" ] 
Amount = [15000, 8000, 6000, 3000,

## Step 9: Build the Prompt

In [9]:
NUM_QUESTIONS = 5

prompt = f"""You are an intelligent educational assistant designed to generate high-quality multiple-choice questions (MCQs) from academic content.

Your task is to generate MCQs ONLY from the provided context. The context is retrieved from a larger document using a retrieval system, so you must strictly rely on it.

Instructions:
1. Carefully read and understand the given context.
2. Generate multiple-choice questions that test conceptual understanding, not just memorization.
3. Each question must have exactly 4 options.
4. Clearly indicate the correct answer.
5. Provide a brief explanation for the correct answer.
6. Avoid duplicate or trivial questions.
7. Ensure questions are clear, concise, and relevant to the context.
8. If the context is insufficient, do NOT hallucinate — instead, return an empty list.
9. Ensure options are plausible and not obviously incorrect.
10. Shuffle correct answers across A, B, C, D positions.
11. Avoid repeating the same answer pattern.
12. Focus on key concepts, definitions, examples, and applications.

Difficulty Levels:
- Easy: Basic definitions or direct facts
- Medium: Conceptual understanding
- Hard: Application-based or tricky questions

Output JSON only in this format:
{{
  "questions": [
    {{
      "question": "Question?",
      "options": ["Option A", "Option B", "Option C", "Option D"],
      "answer": "A",
      "explanation": "Short explanation.",
      "difficulty": "Easy|Medium|Hard"
    }}
  ]
}}

Context:
{retrieved_context}

Generate exactly {NUM_QUESTIONS} questions.
"""

print('Prompt length:', len(prompt), 'chars')

Prompt length: 3699 chars


## Step 10: Call Groq API

In [10]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

response = client.chat.completions.create(
    model='llama-3.3-70b-versatile',
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant that generates multiple choice questions.'},
        {'role': 'user', 'content': prompt}
    ],
    temperature=0.7,
    max_tokens=4000
)

raw = response.choices[0].message.content
print('Raw response length:', len(raw), 'chars')
print(raw[:300])

Raw response length: 2549 chars
```json
{
  "questions": [
    {
      "question": "What is the purpose of the get_side and set_side methods in the Square class?",
      "options": ["To access the private attribute from outside the class", "To modify the private attribute from inside the class", "To create a new instance of the Sq


## Step 11: Parse Response

In [11]:
def parse_mcqs(resp):
    resp = resp.strip()
    # Strip markdown code fences
    resp = re.sub(r'^```json\s*', '', resp)
    resp = re.sub(r'^```\s*', '', resp)
    resp = re.sub(r'\s*```$', '', resp)

    try:
        return json.loads(resp)['questions']
    except (json.JSONDecodeError, KeyError):
        pass

    # Fallback: find JSON object anywhere in string
    match = re.search(r'\{[\s\S]*\}', resp)
    if match:
        return json.loads(match.group(0))['questions']

    raise ValueError('Could not parse MCQ response')


questions = parse_mcqs(raw)
print(f'Parsed {len(questions)} questions')

Parsed 5 questions


## Step 12: Display Results

In [12]:
for i, q in enumerate(questions, 1):
    print(f'Q{i}: {q["question"]}')
    print(f'   Difficulty: {q.get("difficulty", "N/A")}')
    for opt in q['options']:
        print(f'   - {opt}')
    print(f'   Answer: {q["answer"]}')
    print(f'   Explanation: {q.get("explanation", "No explanation provided.")}')
    print()

Q1: What is the purpose of the get_side and set_side methods in the Square class?
   Difficulty: Medium
   - To access the private attribute from outside the class
   - To modify the private attribute from inside the class
   - To create a new instance of the Square class
   - To delete an instance of the Square class
   Answer: A
   Explanation: The get_side and set_side methods are used to access and modify the private attribute 'side' from outside the Square class.

Q2: How would you validate a username to ensure it meets the conditions: only alphabets and digits, does not start with a digit, and length ≥ 6?
   Difficulty: Medium
   - Using a regular expression
   - Using a simple string comparison
   - Using a loop to check each character
   - Using a built-in validation function
   Answer: A
   Explanation: A regular expression can be used to validate the username by checking for the specified conditions.

Q3: What is the purpose of the GetAccountDetails and DisplayAccountDetails 